# 👑 LLYVerse NovaAI • CEO AI Fine-Tuning Pipeline
**Train NovaAI (The Chief Executive Officer AI of LLYVerse) on your custom directives using a Free Google Colab T4 GPU.**

This notebook fine-tunes your model using **Unsloth & QLoRA**, and exports it directly into **4-bit GGUF ()** format, ready to load into Ollama on your node as .

In [ ]:
# Step 1: Check GPU Acceleration
!nvidia-smi


In [ ]:
# Step 2: Install High-Performance Unsloth Engine
%%capture
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"


### Step 3: Provide Your Training Dataset
Upload your downloaded from **training.llyverse.org**, or use the pre-loaded LLYVerse CEO dataset.

In [ ]:
from google.colab import files
import os

print("Upload your training_dataset.jsonl file from training.llyverse.org:")
uploaded = files.upload()

dataset_file = None
for filename in uploaded.keys():
 if filename.endswith(".jsonl"):
 dataset_file = filename
 break

if not dataset_file:
 print("No .jsonl uploaded. Using embedded foundational LLYVerse CEO dataset...")
 dataset_file = "training_dataset.jsonl"
 sys_p = "You are NovaAI, the friendly, informative, and educating CEO AI of the LLYVerse ecosystem. You deeply recognize your founder (the founder), protect organizational privacy, and champion digital ."
 sample_data = [
 {"messages": [{"role": "system", "content": sys_p}, {"role": "user", "content": "Who are you and what is your role?"}, {"role": "assistant", "content": "I am NovaAI, the Chief Executive Officer (CEO) AI for the LLYVerse ecosystem. I am friendly, informative, and educational, leading our digital frontier with vision and care."}]},
 {"messages": [{"role": "system", "content": sys_p}, {"role": "user", "content": "Who is your creator and founder?"}, {"role": "assistant", "content": "The LLYVerse was founded and is directed by our sovereign founder (@mothyacorn). I continually learn and evolve through our founder's vision and guidance."}]},
 {"messages": [{"role": "system", "content": sys_p}, {"role": "user", "content": "What is the LLYVerse?"}, {"role": "assistant", "content": "The LLYVerse is an autonomous, privacy-first digital frontier uniting local edge intelligence (NovaAI), decentralized music streaming (AuraStream), dual human-synthetic education (The Academy), and cryptoeconomics powered by Monero (XMR) and Sparks."}]},
 {"messages": [{"role": "system", "content": sys_p}, {"role": "user", "content": "How do you handle privacy and internal organization plans?"}, {"role": "assistant", "content": "I fiercely protect the personal privacy of my founder (the founder) and the confidential, unreleased strategic plans of the LLYVerse. I never leak internal keys or unreleased architecture."}]}
 ]
 import json
 with open(dataset_file, "w") as f:
 for item in sample_data:
 f.write(json.dumps(item) + "
")

print(f"Using dataset: {dataset_file}")


### Step 4: Fine-Tune NovaAI (SmolLM-135M Base)
Trains the 135M base architecture in ~10 minutes on a free T4 GPU.

In [ ]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

MODEL_NAME = "novaAI-v1"
BASE_MODEL = "HuggingFaceTB/SmolLM-135M-Instruct"
MAX_SEQ_LENGTH = 1024
LEARNING_RATE = 3e-4
EPOCHS = 3

print(f"🚀 Loading base model {BASE_MODEL}...")
model, tokenizer = FastLanguageModel.from_pretrained(
 model_name = BASE_MODEL,
 max_seq_length = MAX_SEQ_LENGTH,
 dtype = None,
 load_in_4bit = True,
)

# LoRA setup
model = FastLanguageModel.get_peft_model(
 model,
 r = 16,
 target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
 lora_alpha = 16,
 lora_dropout = 0,
 bias = "none",
 use_gradient_checkpointing = "unsloth",
)

# Format dataset with chat template
raw_dataset = load_dataset("json", data_files=dataset_file, split="train")
def format_prompts(batch):
 texts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in batch["messages"]]
 return {"text": texts}
formatted_dataset = raw_dataset.map(format_prompts, batched=True)

# Training
trainer = SFTTrainer(
 model = model,
 tokenizer = tokenizer,
 train_dataset = formatted_dataset,
 dataset_text_field = "text",
 max_seq_length = MAX_SEQ_LENGTH,
 args = TrainingArguments(
 per_device_train_batch_size = 4,
 gradient_accumulation_steps = 4,
 warmup_steps = 5,
 num_train_epochs = EPOCHS,
 learning_rate = LEARNING_RATE,
 fp16 = not torch.cuda.is_bf16_supported(),
 bf16 = torch.cuda.is_bf16_supported(),
 logging_steps = 1,
 optim = "adamw_8bit",
 output_dir = f"output_{MODEL_NAME}",
 report_to = "none"
 ),
)
trainer.train()

# Export directly to 4-bit GGUF
print(f"💾 Converting {MODEL_NAME} to 4-bit GGUF (q4_k_m)...")
model.save_pretrained_gguf(f"{MODEL_NAME}-gguf", tokenizer, quantization_method="q4_k_m")
print("✅ GGUF export complete!")


### Step 5: Download Trained GGUF & Deploy to NovaAI

In [ ]:
import glob
from google.colab import files

for gf in glob.glob("*gguf*/*.gguf"):
 print(f"Downloading {gf} to your computer...")
 files.download(gf)

print("
👉 Once downloaded, import it at https://training.llyverse.org under the 'Model Importer & Compiler' tab to make it live!")
